# Введение в регрессионный анализ

*Алла Тамбовцева*

## Практикум 6. Линейные модели с фиктивными переменными

In [1]:
import pandas as pd
import statsmodels.formula.api as smf

### Сюжет 1: изучаем новости науки

С сайта [nplus1.ru](https://nplus1.ru/), где публикуются новости науки, были выгружены данные по статьям за 2026 год.

Переменные в файле [nplus_one.csv](https://github.com/allatambov/PyReg26/blob/main/nplus_one.csv):

* `title`: заголовок новости;		
* `author`: автор;
* `date`: дата публикации;
* `diffc`: сложность новости;
* `rubrics`: рубрики (тематические разделы);
* `text`: полный текст новости.

Для дальнейшей работы нужно выбрать новости, которые относятся к **трем самым популярным рубрикам** (то есть с наибольшим количеством новостей). Также нужно выяснить, кто из авторов написал самую сложную статью.

#### Задачи

1. На отобранных данных необходимо выяснить, есть ли значимые различия в сложности новостей, которые пишет автор самой сложной статьи, и новостей, написанных другими авторами.

2. На отобранных данных необходимо построить модель множественной линейной регрессии, которая объясняет, каким образом длина новости (число символов в полном тексте) и ее тематика сказываются на сложности новости.

#### Подготовка данных

Загрузим данные и выведем первые пять строк датафрейма для наглядности:

In [2]:
df = pd.read_csv("nplus_one.csv")
df.head()

,Unnamed: 0,title,author,date,diffc,rubrics,text
0,0,Температуру ядра Земли уточнили с помощью лазе...,Егор Конюхов,2025-01-03,2.9,"Физика, Геология",Физики уточнили температуру внутренней границы...
1,1,Большинство людей эпохи мезолита из Оленеостро...,Михаил Подрезов,2025-01-03,3.1,"Археология, Антропология",Ученые проанализировали изотопный состав строн...
2,2,Физики разобрались в состоянии покоя ткани. Он...,Егор Конюхов,2025-01-03,2.3,Физика,"Группа физиков выяснила, что при отсутствии вн..."
3,3,Виноградины усилили микроволновое поле и помог...,Егор Конюхов,2025-01-06,4.1,Физика,Физики использовали две виноградины для усилен...
4,4,Прием глюкокортикоидов при беременности повыси...,Олег Лищук,2025-01-06,1.2,"Медицина, Психология",Кристина Лаугесен (Kristina Laugesen) из Орхус...


Чтобы узнать, какие рубрики являются самыми популярными – с наибольшим числом новостей – выведем таблицу частот для столбца `rubrics`:

In [3]:
df["rubrics"].value_counts()

rubrics
Медицина                       1076
Археология                      311
Зоология                        304
Антропология, Археология        157
Астрономия                      153
                               ... 
Материалы, Медицина, Физика       1
IT, Лингвистика                   1
Биология, Мезонин                 1
IT, Наука                         1
Астрономия, IT                    1
Name: count, Length: 275, dtype: int64

Результат выше – последовательность `pandas Series`, то есть отдельный столбец с числовыми значениями (количество новостей) и текстовыми индексами (названия рубрик). Выберем три первые элемента и извлечем их индексы:

In [4]:
names = df["rubrics"].value_counts().head(3).index
print(names)

Index(['Медицина', 'Археология', 'Зоология'], dtype='str', name='rubrics')


Вспомним про метод `.isin()`, который позволяет проверить, входит ли значение в список, и построим с ним фильтр для отбора новостей, относящихся к рубрикам из `names`:

In [5]:
small = df[df["rubrics"].isin(names)]
small

,Unnamed: 0,title,author,date,diffc,rubrics,text
5,5,Уровень лейкоцитов в крови суданской пациентки...,Олег Лищук,2025-01-06,2.2,Медицина,В суданскую больницу поступила 64-летняя женщи...
9,9,Напитки с сахаром назвали причиной почти 10 пр...,Олег Лищук,2025-01-07,1.9,Медицина,Американские и мексиканские исследователи с ме...
14,14,Новые рекомендации сократили сроки лечения туб...,Олег Лищук,2025-01-08,3.9,Медицина,Юсси Сауконнен (Jussi Saukkonen) из Бостонског...
15,15,Врачи вылечили новорожденного с врожденной гры...,Олег Лищук,2025-01-08,2.8,Медицина,Американские врачи сообщили об успешном случае...
16,16,Среди частых побочных эффектов CAR-T-терапии в...,Олег Лищук,2025-01-08,3.3,Медицина,Эрина Хосоя (Erina Hosoya) с коллегами по Унив...
...,...,...,...,...,...,...,...
3698,1921,Кошки замяукали в присутствии хозяина и при ви...,Катерина Петрова,2026-11-12,1.5,Зоология,"Бразильские ученые обнаружили, что мяуканье до..."
3699,1922,Отмена гипотензивных препаратов утром перед оп...,Слава Гоменюк,2026-11-12,2.2,Медицина,Прекращение приема ингибиторов ангиотензинпрев...
3700,1923,Положение при мочеиспускании у мужчин существе...,Слава Гоменюк,2026-11-12,2.1,Медицина,У здоровых мужчин поза при мочеиспускании — ст...
3704,1927,Миниатюрные клопы нового вида провели всю жизн...,Сергей Коленов,2026-11-12,3.7,Зоология,Энтомологи обнаружили на юго-востоке Китая нов...


Определим автора, который написал самую сложную новость:

In [6]:
# выберем строку, где сложность максимальная
# автор – Юлия Нестерова

small[small["diffc"] == small["diffc"].max()]

,Unnamed: 0,title,author,date,diffc,rubrics,text
307,307,Вирус герпеса повысил риск болезни Альцгеймера...,Юлия Нестерова,2025-02-28,7.2,Медицина,Ученые из Норвегии и США изучили образцы ткане...


#### Построение модели 1

Чтобы сравнить средние уровни сложности новостей, написанных Юлией Нестеровой (автор самой сложной новости), и всеми остальными авторами, нужно построить парную линейную регрессию, где зависимая переменная – сложность новости, а объясняющая – бинарный признак, где 1 соответствует новостям от Юлии Нестеровой.

Создадим такой столбец из 0 и 1:

* проверим условие `small["author"] == "Юлия Нестерова"`
* получим столбец из `True` и `False` (логический тип `bool`)
* переведем тип столбца в целочисленный (тип `int`)

In [7]:
small["da"] = (small["author"] == "Юлия Нестерова").astype(int)
small["da"].value_counts() # проверяем значения

da
0    1681
1      10
Name: count, dtype: int64

Теперь все готово к построению модели. Вызываем функцию `ols()` и записываем уравнение парной регрессии:

In [8]:
m1 = smf.ols("diffc ~ da", data = small).fit()
print(m1.summary())

                            OLS Regression Results                            
Dep. Variable:                  diffc   R-squared:                       0.047
Model:                            OLS   Adj. R-squared:                  0.046
Method:                 Least Squares   F-statistic:                     83.02
Date:                Fri, 20 Nov 2026   Prob (F-statistic):           2.23e-19
Time:                        02:28:12   Log-Likelihood:                -2018.9
No. Observations:                1691   AIC:                             4042.
Df Residuals:                    1689   BIC:                             4053.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      2.3111      0.019    118.600      0.0

**Уравнение модели:**

$$
\widehat{diffc}_i = 2.311 + 2.308 \times da_i
$$

**Интерпретация:**

* Средняя сложность новостей, написанных остальными авторами (кроме Юлии Нестеровой), равна 2.31.
* Средняя сложностей новостей, написанных Юлией Нестеровой, на 2.308 превышает среднюю сложность новостей от других авторов, а значит, составляет 2.31 + 2.308 $\approx$ 4.62. Различия в средних значениях сложности в этих двух группах авторов являются статистически значимыми на уровне значимости 5% (p-value примерно 0, наблюдаемое значение t-статистики выходит за границы отрезка [-2, 2], 95%-ный доверительный интервал не включает 0).
* $R^2$ у модели достаточно низкий, `R-squared: 0.047`, но это не говорит в данном случае о плохом качестве модели. У парной модели, где независимая переменная – бинарная, обычно и не бывает высокого значения $R^2$ из-за низкой изменчивости бинарных показателей (не может быть большого разнообразия по определению, всего два значения, плюс, вспомните как выглядит дисперсия бинарной случайной величины).

#### Построение модели 2

Для построения модели множественной линейной регрессии, которая объясняет, каким образом длина новости (число символов в полном тексте) и ее тематика сказываются на сложности новости, надо добавить столбец с длиной новости.

Вспомним про метод `str.len()`, который считает число символов в каждой ячейке столбца:

In [9]:
small["text_len"] = small["text"].str.len()
small.head() # 5 строк для примера

,Unnamed: 0,title,author,date,diffc,rubrics,text,da,text_len
5,5,Уровень лейкоцитов в крови суданской пациентки...,Олег Лищук,2025-01-06,2.2,Медицина,В суданскую больницу поступила 64-летняя женщи...,0,1861
9,9,Напитки с сахаром назвали причиной почти 10 пр...,Олег Лищук,2025-01-07,1.9,Медицина,Американские и мексиканские исследователи с ме...,0,3283
14,14,Новые рекомендации сократили сроки лечения туб...,Олег Лищук,2025-01-08,3.9,Медицина,Юсси Сауконнен (Jussi Saukkonen) из Бостонског...,0,1404
15,15,Врачи вылечили новорожденного с врожденной гры...,Олег Лищук,2025-01-08,2.8,Медицина,Американские врачи сообщили об успешном случае...,0,1436
16,16,Среди частых побочных эффектов CAR-T-терапии в...,Олег Лищук,2025-01-08,3.3,Медицина,Эрина Хосоя (Erina Hosoya) с коллегами по Унив...,0,1090


По идее, чтобы учесть в линейной модели рубрики, к которым относятся новости (нечисловой признак), нужно:

* создать набор фиктивных переменных (по одной для каждой рубрики);
* выбрать, какую рубрику мы возьмем за базовую категорию;
* добавить в модель в качестве независимых переменных все фиктивные переменные, кроме той, которая относится к базовой категории.

Другими словами, получить что-то такое:

| rubrics    |   Археология |   Зоология |   Медицина |
|-----------:|-------------:|-----------:|-----------:|
|  Медицина  |            0 |          0 |          1 |
|  Медицина  |            0 |          0 |          1 |
| Археология |            1 |          0 |          0 |
| Зоология   |            0 |          1 |          0 |

А затем принять рубрику `Археология` за базовую категорию (с ней будем сравнивать остальные) и добавить в модель столбцы `Зоология` и `Медицина`. Однако со `statsmodels` все просто – функция `ols()` сделает эти операции за нас, если в уравнении мы укажем текстовый столбец!

По умолчанию в качестве базовой категории будет выбрана рубрика, которая идет первой по алфавиту:

In [10]:
m2 = smf.ols("diffc ~ text_len + rubrics", data = small).fit()
print(m2.summary())

                            OLS Regression Results                            
Dep. Variable:                  diffc   R-squared:                       0.465
Model:                            OLS   Adj. R-squared:                  0.464
Method:                 Least Squares   F-statistic:                     488.8
Date:                Fri, 20 Nov 2026   Prob (F-statistic):          1.58e-228
Time:                        02:28:13   Log-Likelihood:                -1530.6
No. Observations:                1691   AIC:                             3069.
Df Residuals:                    1687   BIC:                             3091.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                          coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------
Intercept               0.9843    

**Уравнение модели:**

$$
\widehat{diffc}_i = 0.98 + 0.09 \times \text{Зоология}_i + 0.18 \times \text{Медицина}_i + 0.00 \times \text{text\_len}_i
$$

В выдаче `T` в `rubrics[T.Зоология]` и `rubrics[T.Медицина]` означает `True`, то есть это бинарные столбцы, где 1 соответствует рубрикам *Зоология* и *Медицина* соответственно. Базовая категория – `Археология`, ее нет в выдаче, но мы знаем, что это третья рубрика в `small`.

**Интерпретация:**

* `Intercept`: cредняя сложность новостей рубрики *Археология* длиной 0 символов (`text_len = 0`) равна 0.98, но раз новостей длины 0 не бывает, константу нет смысла интерпретировать.
* `rubrics[T.Зоология]`: средняя сложность новостей рубрики *Зоология*, при прочих равных, на 0.09 выше сложности новостей рубрики *Археология* (при прочих равных здесь – если сравниваем новости одинаковой длины, но из разных рубрик, так как кроме рубрик и длины `text_len` в модели больше ничего нет).
* `rubrics[T.Медицина]`: средняя сложность новостей рубрики *Медицина*, при прочих равных, на 0.18 выше сложности новостей рубрики *Археология*.
* Все коэффициенты значимо отличаются от 0 на 5%-ном уровне значимости, средняя сложность новостей в разных рубриках действительно отличается.
* Коэффициент при длине текста, формально тоже статистически значим, однако его величина очень маленькая, почти нулевая. Это ожидаемо: коэффициент показывает, при прочих равных (при сравнении новостей из одной и той же рубрики), на сколько, с среднем, изменяется сложность новости при увеличении ее длины на *один символ*. Логичнее будет пересмотреть подход, например, считать длину текста хотя бы в словах или вычислять, например, среднюю длину слов после удаления незначимых коротких слов (союзы, предлоги, частицы).
* $R^2$ у модели составляет 0.465, модель объясняет 47% дисперсии зависимой переменной, что неплохо, особенно, если учесть, что две независимые переменные – бинарные, то есть с заведомо невысокой изменчивостью.

### Сюжет 2: реплицируем исследование на стыке политологии и экономики

В исследовании Д.Райта *Aid Effectiveness and the Politics of Personalism* (2010) сравнивается эффект иностранной помощи на экономический рост в государствах с низким и высоким уровнем политического персонализма.

По [ссылке](https://disk.yandex.ru/d/sAbsQ59nMou3ZA) доступны данные для репликации, а также сама статья. Кроме того, данные можно нати в открытом доступе на сайте проекта [AidData](https://www.aiddata.org/replication-datasets/aid-effectiveness-and-the-politics-of-personalism).

#### Задача

Изучить таблицу 1 из статьи и do-файл с кодом Stata, выделить две подвыборки и воспроизвести модели, построенные автором для этой таблицы.

#### Понимаем автора (кроме названий столбцов, тут сложно понять и простить)

Рассмотрим фрагмент do-файла:

    reg  gdpg  lgdp ssa  easia  _Iper* l1234natpc  if democracy==1  & personalism<5.2
    reg  gdpg  lgdp ssa  easia _Iper* l1234natpc  if democracy==1  & personalism>5.2 & personalism~=.

Есть кратко, автор делает следующее:

* делит данные на две подвыборки (демократии с уровнем персонализма выше 5.2 и ниже 5.2);
* запускает две множественные линейные регрессии с фиктивными переменными на период `_Iper*`.

Зависимая переменная – экономический рост на основе ВВП (`gdpg`), независимые переменные: 

* иностранная помощь в виде инвестиций (`l1234natpc`),
* бинарная переменная на регион *Африка к югу от Сахары* (`ssa`, 1 для Sub-Saharian Africa)
* бинарная переменная на регион *Восточная Азия* (`easia`, 1 для East Asia)
* логарифмированный ВВП за прошлые периоды (`lgdp`).

Чтобы понять, что это за фиктивные переменные на период, загрузим данные и посмотрим на таблицу:

In [11]:
dat = pd.read_stata("aidpers.dta")
dat.head()

,countrycode,country,year,period,periodstart,periodend,code,code2,democracy,polity2,...,trade_gdp,L_PC_PRODUCTION,coldwar,invest4ave,govtc4ave,egypt,lgdp2t,demhistory,begindem,icrg4ave
0,AFG,Afghanistan,1961.0,-1,1958,1961,AFG-1,AFG58,NaN,NaN,...,NaN,NaN,1.0,NaN,NaN,0,NaN,NaN,NaN,NaN
1,AFG,Afghanistan,1965.0,0,1962,1965,AFG0,AFG62,NaN,NaN,...,NaN,NaN,1.0,NaN,NaN,0,NaN,NaN,NaN,NaN
2,AFG,Afghanistan,1969.0,1,1966,1969,AFG1,AFG66,NaN,NaN,...,NaN,NaN,1.0,NaN,NaN,0,NaN,NaN,NaN,NaN
3,AFG,Afghanistan,1973.0,2,1970,1973,AFG2,AFG70,NaN,NaN,...,NaN,NaN,1.0,NaN,NaN,0,NaN,NaN,NaN,NaN
4,AFG,Afghanistan,1977.0,3,1974,1977,AFG3,AFG74,NaN,NaN,...,NaN,NaN,1.0,NaN,NaN,0,NaN,NaN,NaN,NaN


Автор просто объединил годы наблюдений в периоды по два года и закодировал их (см. столбцы `period`, `periodstart` и	`periodend`), начиная с -1. Посмотрим на значения в `period`:

In [12]:
dat["period"].value_counts()

period
 7    129
 8    129
 9    129
 6    108
 5    107
 4    106
 3    104
 2    101
 1    100
 0     94
-1     92
Name: count, dtype: int64

Чтобы корректно учесть период в модели, вместо `period` (это же не полноценные числа, которые позволяют что-то померить, а условные коды) в нее нужно добавлять набор фиктивных переменных на каждый период, приняв за базовую категорию самый ранний период с кодом -1.

#### Делим данные на группы и строим модели

Сохраним демократии с низким уровнем политического персонализма в `low`, с высоким – в `high`:

In [13]:
low = dat[(dat["democracy"] == 1) & (dat["personalism"] < 5.2)]
high = dat[(dat["democracy"] == 1) & (dat["personalism"] > 5.2)]

Оценим модель множественной регрессии как обычно, подав на вход уравнение, только внесем в него одно изменение – вокруг периода добавим функцию `C()`. Эта функция сообщит `ols()`, что к столбцу с периодом нужно относиться как к текстовому (качественному, категориальному, отсюда `C` от *categorical*), а не как к числовому. 

Без `C()` функция `ols()` будет считать номер периода за полноценную числовую переменную и оценивать коэффициент при ней как среднее изменение зависимой переменной при увеличении номера периода на 1, что некорректно. С `C()` функция `ols()` создаст дамми-переменные на каждый период (как ранее на каждую рубрику новости) и добавит их в модель (кроме первого периода с кодом -1 – базовой категории).

In [14]:
mod1 = smf.ols("gdpg ~ lgdp + ssa + easia + l1234natpc + C(period)", data = low).fit()
print(mod1.summary())

                            OLS Regression Results                            
Dep. Variable:                   gdpg   R-squared:                       0.139
Model:                            OLS   Adj. R-squared:                  0.043
Method:                 Least Squares   F-statistic:                     1.446
Date:                Fri, 20 Nov 2026   Prob (F-statistic):              0.149
Time:                        02:28:15   Log-Likelihood:                -308.52
No. Observations:                 130   AIC:                             645.0
Df Residuals:                     116   BIC:                             685.2
Df Model:                          13                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept         -3.9214      3.030     -1.

In [15]:
mod2 = smf.ols("gdpg ~ lgdp + ssa + easia + l1234natpc + C(period)", data = high).fit()
print(mod2.summary())

                            OLS Regression Results                            
Dep. Variable:                   gdpg   R-squared:                       0.140
Model:                            OLS   Adj. R-squared:                  0.042
Method:                 Least Squares   F-statistic:                     1.436
Date:                Fri, 20 Nov 2026   Prob (F-statistic):              0.154
Time:                        02:28:15   Log-Likelihood:                -311.88
No. Observations:                 129   AIC:                             651.8
Df Residuals:                     115   BIC:                             691.8
Df Model:                          13                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept          1.2887      2.739      0.

В таблице 1 в статье представлены результаты этих моделей, только из нее исключены оценки коэффициентов для периодов, поскольку они не находятся в фокусе анализа, а, скорее, добавляются для реалистичности модели – экономический рост и объем иностранной помощи в течение времени изменялись, то есть отличались от периода к периоду. Значения стандартных ошибок и p-value здесь отличаются от тех, что в таблице, так как автор использовал специальный вид стандартных ошибок – скорректированных с учетом пространственно-временной структуры данных (не просто наблюдения, а наблюдения, которые можно сгруппировать по странам или по годам).